# Notebook 01 - Exploration des donnees

**Projet** : MixCraft - Systeme Generatif Intelligent pour la Creation de Cocktails  
**Auteurs** : Adam Beloucif & Emilien Morice  
**EFREI Paris - M1 Data Engineering & IA - Decembre 2025**

---

## Objectifs

1. Charger et fusionner les 4 datasets Kaggle
2. Analyser la structure et la qualite des donnees
3. Produire des visualisations exploratoires (distributions, word clouds)
4. Identifier les features pertinentes pour la recommandation et la generation

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Config plot
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
EFREI_NAVY = '#163767'
EFREI_PINK = '#FF43B8'
EFREI_BLUE = '#0C78B4'

print('Imports OK')

## 1. Chargement des datasets

In [ ]:
from src.data_loader import load_all_datasets, load_main_dataset, load_iba_dataset

# Chargement du corpus complet (avec fallback donnees synthetiques)
df = load_all_datasets()

print(f'Corpus total : {len(df)} cocktails')
print(f'Colonnes : {list(df.columns)}')
df.head()

In [ ]:
# Statistiques generales
print('=== Statistiques generales ===')
print(f'Nombre de cocktails         : {len(df)}')
print(f'Categories distinctes       : {df["category"].nunique()}')
print(f'Types de verre              : {df["glass"].nunique()}')
print(f'Sources de donnees          : {df["source"].nunique() if "source" in df.columns else "N/A"}')
print(f'Valeurs manquantes (name)   : {df["name"].isna().sum()}')
print(f'Valeurs manquantes (ing)    : {df["ingredients"].isna().sum()}')
print(f'Valeurs manquantes (instr.) : {df["instructions"].isna().sum()}')

## 2. Distribution des categories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Categories
cat_counts = df['category'].value_counts()
colors = [EFREI_NAVY if i == 0 else EFREI_BLUE if i == 1 else '#a8bdd4'
          for i in range(len(cat_counts[:10]))]

axes[0].barh(cat_counts[:10].index[::-1], cat_counts[:10].values[::-1], color=colors[::-1])
axes[0].set_title('Top 10 categories de cocktails', fontweight='bold', color=EFREI_NAVY)
axes[0].set_xlabel('Nombre de cocktails')

# Types de verre
glass_counts = df['glass'].value_counts().head(8)
wedge_colors = [EFREI_NAVY, EFREI_BLUE, EFREI_PINK, '#0C78B4', '#a8bdd4', '#FFB3E0', '#4a90c4', '#2d6ea8']
axes[1].pie(
    glass_counts.values,
    labels=[g[:20] for g in glass_counts.index],
    colors=wedge_colors[:len(glass_counts)],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 9}
)
axes[1].set_title('Repartition par type de verre', fontweight='bold', color=EFREI_NAVY)

plt.tight_layout()
plt.savefig('../assets/fig_categories.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardee : assets/fig_categories.png')

## 3. Analyse des ingredients

In [ ]:
import re

# Extraction des ingredients individuels
all_ings = (
    df['ingredients']
    .fillna('')
    .str.lower()
    .str.split(r'[,\n]')
    .explode()
    .str.strip()
    .str.replace(r'\d+(?:ml|cl|oz|tsp|tbsp)?\s*', '', regex=True)
    .str.strip()
)
all_ings = all_ings[all_ings.str.len() > 2]
ing_counts = all_ings.value_counts()

print(f'Ingredients uniques : {len(ing_counts)}')
print('\nTop 20 ingredients :')
print(ing_counts.head(20).to_string())

In [ ]:
# Word cloud des ingredients
try:
    from wordcloud import WordCloud
    wc_text = ' '.join(all_ings.tolist())
    wc = WordCloud(
        width=900, height=450,
        background_color='white',
        colormap='Blues',
        max_words=80,
        prefer_horizontal=0.8,
    ).generate(wc_text)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud des ingredients', fontsize=16, fontweight='bold', color=EFREI_NAVY, pad=15)
    plt.tight_layout()
    plt.savefig('../assets/fig_wordcloud.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print('wordcloud non installe (pip install wordcloud)')
    # Fallback : bar chart
    top30 = ing_counts.head(30)
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.barh(top30.index[::-1], top30.values[::-1], color=EFREI_BLUE)
    ax.set_title('Top 30 ingredients', fontweight='bold', color=EFREI_NAVY)
    plt.tight_layout()
    plt.savefig('../assets/fig_ingredients.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Profils de saveurs

In [ ]:
flavor_cols = ['sweet', 'sour', 'bitter', 'strong', 'fresh', 'fruity']
available_flavors = [c for c in flavor_cols if c in df.columns]

if available_flavors:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for i, col in enumerate(available_flavors):
        axes[i].hist(df[col], bins=15, color=EFREI_NAVY, alpha=0.8, edgecolor='white')
        axes[i].set_title(f'Distribution : {col}', fontweight='bold', color=EFREI_NAVY)
        axes[i].set_xlabel('Score [0, 1]')
        axes[i].set_ylabel('Frequence')
        mean_val = df[col].mean()
        axes[i].axvline(mean_val, color=EFREI_PINK, linestyle='--', label=f'Moyenne : {mean_val:.2f}')
        axes[i].legend(fontsize=9)

    plt.suptitle('Distributions des profils de saveurs', fontsize=16, fontweight='bold', color=EFREI_NAVY, y=1.01)
    plt.tight_layout()
    plt.savefig('../assets/fig_flavors.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Colonnes de saveurs non disponibles.')

In [ ]:
# Matrice de correlation des saveurs
if available_flavors:
    corr = df[available_flavors].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, annot=True, fmt='.2f',
        cmap='Blues', ax=ax, vmin=-1, vmax=1,
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title('Correlation entre les profils de saveurs', fontweight='bold', color=EFREI_NAVY)
    plt.tight_layout()
    plt.savefig('../assets/fig_corr_flavors.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Longueur des instructions

In [ ]:
df['instructions_len'] = df['instructions'].fillna('').str.len()
df['ingredients_count'] = df['ingredients'].fillna('').str.count(',') + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['instructions_len'], bins=30, color=EFREI_BLUE, edgecolor='white', alpha=0.9)
axes[0].set_title('Longueur des instructions (nb caracteres)', fontweight='bold', color=EFREI_NAVY)
axes[0].set_xlabel('Caracteres')
axes[0].set_ylabel('Frequence')
axes[0].axvline(df['instructions_len'].median(), color=EFREI_PINK, linestyle='--',
                label=f'Mediane : {df["instructions_len"].median():.0f}')
axes[0].legend()

axes[1].hist(df['ingredients_count'], bins=range(1, 20), color=EFREI_NAVY, edgecolor='white', alpha=0.9)
axes[1].set_title('Nombre d\'ingredients par recette', fontweight='bold', color=EFREI_NAVY)
axes[1].set_xlabel('Nombre d\'ingredients')
axes[1].set_ylabel('Frequence')
axes[1].axvline(df['ingredients_count'].median(), color=EFREI_PINK, linestyle='--',
                label=f'Mediane : {df["ingredients_count"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../assets/fig_text_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mediane instructions : {df["instructions_len"].median():.0f} caracteres')
print(f'Mediane ingredients  : {df["ingredients_count"].median():.1f}')

## 6. Sauvegarde du corpus nettoye

In [ ]:
from pathlib import Path
processed_dir = Path('../data/processed')
processed_dir.mkdir(exist_ok=True)

df.to_csv(processed_dir / 'corpus_cocktails.csv', index=False)
print(f'Corpus sauvegarde : data/processed/corpus_cocktails.csv ({len(df)} lignes)')

# Resume EDA
print('\n=== Resume EDA ===')
print(f'  Cocktails total        : {len(df)}')
print(f'  Categories             : {df["category"].nunique()}')
print(f'  Ingredients uniques    : {len(ing_counts)}')
print(f'  Moy. ingredients/recto : {df["ingredients_count"].mean():.1f}')
print(f'  Moy. longueur instr.   : {df["instructions_len"].mean():.0f} chars')